# Stage 2 — Threshold decision, cleaning, features and leakage tests

| Part | Prompt | Output |
|---|---|---|
| 0 | 1.2 — choose the rain-day threshold | decision + Methods paragraph |
| 1 | 2.1 — cleaning | `data/processed/clean_daily.parquet` |
| 2 | 2.2 — targets and features | `data/processed/features_A.parquet`, `features_B.parquet`, feature dictionary |
| 3 | 2.3 — leakage tests | truncation, target-shift and correlation tests |

The feature code lives in `src/features.py` so that this notebook, the leakage tests and later notebooks all use the **same**
functions (rule 7: identical features for every model). Approved plan: see DECISIONS.md entries 1.1–1.10 and 2.x.

**Training rows (for any statistic below that could influence modelling):** rows whose *target day* is in 1981–2019,
i.e. feature date ≤ 2019-12-30. The row dated 2019-12-31 predicts 2020-01-01, which belongs to the test period.

### Step 1 — Setup

**WHAT:** imports, paths, seed; make `src/` importable; load the Stage-0 parsed file.

**WHY:** rule 15 — this notebook reads only from `data/processed` and writes its outputs to disk.

**Alternatives considered:** installing `src` as a package (`pip install -e .`) — cleaner for a larger project; a
`sys.path` entry is enough here and keeps the setup visible.

**Look for:** library versions and the parsed data shape (16,436 × 15).

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from sklearn.metrics import cohen_kappa_score

SEED = 42; rng = np.random.default_rng(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import features as F

PROC, FIG, TAB = ROOT / "data/processed", ROOT / "reports/figures", ROOT / "reports/tables"
TRAIN_LAST_ROW = "2019-12-30"            # last feature date whose target (t+1) is still in 2019
SEASON = {12: "winter", 1: "winter", 2: "winter", 3: "pre-monsoon", 4: "pre-monsoon", 5: "pre-monsoon",
          6: "monsoon", 7: "monsoon", 8: "monsoon", 9: "monsoon", 10: "post-monsoon", 11: "post-monsoon"}
pd.set_option("display.width", 160, "display.max_columns", 30, "display.precision", 3)

parsed = pd.read_parquet(PROC / "power_parsed.parquet")
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| parsed:", parsed.shape)

pandas 3.0.5 | scikit-learn 1.9.1 | parsed: (16436, 15)


**Interpretation:** environment loads (pandas 3.0.5, scikit-learn 1.9.1); the parsed file has the expected 16,436 × 15.

**Decision:** `src/features.py` is the single source of truth for feature and target definitions.

**Paper note:** *Feature construction was implemented as a version-controlled Python module shared by all experiments to
guarantee identical inputs across models.*

# Part 0 — Prompt 1.2: choosing the rain-day threshold

### Step 2 — Evidence for 0.1, 1 and 2.5 mm

**WHAT:** for each candidate threshold, using training years only, compute:
1. **class balance** — share of rain days overall, and the most extreme seasonal shares;
2. **trace-value noise** — share of all days with 0 < rain < threshold (tiny gridded values the threshold calls "dry") and
   share of *rain days* that lie in [threshold, 1 mm) — i.e. days counted as rain only because of drizzle-scale amounts;
3. **agreement with IMERG** on the same day (1998–2019): percentage agreement and **Cohen's κ** (agreement corrected for
   chance). Two independent products agreeing on "rain" suggests the threshold captures real events, not product noise;
4. **sensitivity to the 2016 regime shift** — rain-day share in 1998–2015 vs 2016–2019.

**WHY:** the threshold defines the classification task. A good threshold gives a reasonably balanced, physically
meaningful and robust target that other studies also use.

**Alternatives considered:** a data-driven threshold (e.g. the one maximising IMERG agreement) — optimising the definition
of the target on our own data is a form of researcher degrees of freedom; we use conventional thresholds and report all three.

**Look for:** where κ is highest and which threshold is least sensitive to the 2016 shift.

In [2]:
tr = parsed.loc[:"2019"]
common = tr.loc["1998":, ["PRECTOTCORR", "IMERG_PRECTOT"]]
season = tr.index.month.map(SEASON)
rows = {}
for t in (0.1, 1.0, 2.5):
    wet = tr["PRECTOTCORR"] >= t
    by_season = wet.groupby(season).mean()
    p, i = common["PRECTOTCORR"] >= t, common["IMERG_PRECTOT"] >= t
    rows[f">= {t} mm"] = {
        "rain_day_share": wet.mean(),
        "min_season_share": f"{by_season.min():.2f} ({by_season.idxmin()})",
        "max_season_share": f"{by_season.max():.2f} ({by_season.idxmax()})",
        "share_days_trace_below_thr": ((tr["PRECTOTCORR"] > 0) & ~wet).mean(),
        "share_raindays_below_1mm": ((tr["PRECTOTCORR"] < 1) & wet).sum() / wet.sum(),
        "agree_IMERG_pct": (p == i).mean() * 100,
        "kappa_IMERG": cohen_kappa_score(p, i),
        "share_1998_2015": wet.loc["1998":"2015"].mean(),
        "share_2016_2019": wet.loc["2016":"2019"].mean(),
    }
thr_tab = pd.DataFrame(rows).T
thr_tab["shift_2016_pct_points"] = (thr_tab["share_2016_2019"] - thr_tab["share_1998_2015"]) * 100
thr_tab.to_csv(TAB / "02_threshold_evidence.csv")
thr_tab

,rain_day_share,min_season_share,max_season_share,share_days_trace_below_thr,share_raindays_below_1mm,agree_IMERG_pct,kappa_IMERG,share_1998_2015,share_2016_2019,shift_2016_pct_points
>= 0.1 mm,0.594,0.14 (winter),0.96 (monsoon),0.055,0.169,87.617,0.748,0.596,0.65,5.372
>= 1.0 mm,0.494,0.07 (winter),0.88 (monsoon),0.155,0.0,87.492,0.75,0.505,0.589,8.43
>= 2.5 mm,0.422,0.05 (winter),0.78 (monsoon),0.226,0.0,85.737,0.711,0.444,0.547,10.241


**Interpretation — pros and cons for this dataset:**

| | **0.1 mm** | **1.0 mm** | **2.5 mm** |
|---|---|---|---|
| Rain-day share (train) | 59.4 % | **49.4 % (balanced)** | 42.2 % |
| Seasonal range | 14 % (winter) – 96 % (monsoon) | 7 % – 88 % | 5 % – 78 % |
| Drizzle noise | **17 % of its rain days are < 1 mm** — mostly reanalysis trace amounts, where products disagree most (Stage 1: zero-share 35 % vs 43 %) | trace days (15.5 % of all days) count as dry | 22.6 % of days with some rain count as dry; discards light but real rain |
| Same-day agreement with IMERG, κ | 0.748 | **0.750** | 0.711 |
| Shift in rain-day share, 1998–2015 → 2016–2019 | +5.4 pp | +8.4 pp | +10.2 pp |
| Literature | "measurable precipitation" / gauge trace level; rarely used for gridded data | **ETCCDI wet-day definition (R1mm, SDII)**; standard in "rain tomorrow" ML studies | **India Meteorological Department "rainy day"**, common in South-Asian climatology |

- **1.0 mm** gives the most balanced classes, the best agreement with the independent satellite product (tied with 0.1 mm),
  removes the reanalysis drizzle that makes 0.1 mm product-dependent, and matches the international climate-index standard.
- **Honest caveat:** no threshold escapes the 2016 shift. 0.1 mm is the *least* sensitive to it (+5.4 pp), 1.0 mm is in the
  middle (+8.4 pp), 2.5 mm the most (+10.2 pp). This is another reason to keep 0.1 and 2.5 mm as sensitivity analyses.

**Decision:** **primary threshold = 1.0 mm (`y_cls`)**; **0.1 mm (`y_cls_01`) and 2.5 mm (`y_cls_25`) as sensitivity
analyses**, reported for the best models and the baselines. (Citations: ETCCDI indices — Zhang et al., 2011, *WIREs Climate
Change*; IMD rainy-day definition — IMD glossary. Please verify the exact references before submission.)

**Methods paragraph (draft):**
> *A rain day was defined as a day with at least 1.0 mm of PRECTOTCORR precipitation, following the wet-day definition of
> the ETCCDI climate indices. This threshold yields an approximately balanced binary target over the training period
> (49.4 % rain days, 1981–2019), filters the trace amounts (< 1 mm) that are characteristic of reanalysis precipitation and
> differ most between products, and gave the highest same-day agreement with GPM IMERG among the candidate thresholds
> (Cohen's κ = 0.75, 1998–2019). To assess the sensitivity of our conclusions to this choice, all classification
> experiments were repeated with thresholds of 0.1 mm (any measurable precipitation; 59.4 % rain days) and 2.5 mm (the rainy-day
> definition commonly used in South-Asian climatology; 42.2 % rain days).*

**Paper note:** *The 1.0 mm threshold balanced class frequencies, robustness to trace precipitation and comparability with
established climate indices; 0.1 and 2.5 mm were retained as sensitivity thresholds.*

# Part 1 — Prompt 2.1: cleaning

### Step 3 — Missing-value code and duplicate dates

**WHAT:** re-apply `-999 → NaN` (idempotent: it changes nothing if Stage 0 already did it), and verify that no `-999` remains
and that there are 0 duplicate dates; drop duplicates only if any exist.

**WHY:** the cleaning step must be complete on its own, so the plan's first two items are executed and *verified* here
rather than assumed.

**Alternatives considered:** skipping (Stage 0 already did it) — but an explicit assertion documents the guarantee.

**Look for:** `-999 remaining: 0`, `duplicate dates: 0`.

In [3]:
clean = parsed.replace(-999, np.nan)
print("-999 remaining:", int((clean == -999).sum().sum()))
dup = clean.index.duplicated()
print("duplicate dates:", dup.sum())
clean = clean[~dup]
assert clean.index.is_monotonic_increasing and pd.infer_freq(clean.index) == "D"

-999 remaining: 0
duplicate dates: 0


**Interpretation:** no `-999` remains and no duplicate dates exist; the index is a strictly increasing daily index.
Both plan items are verified, and nothing had to be changed.

**Decision:** proceed.

**Paper note:** *After conversion of the source missing-value code, the record contained no duplicated dates.*

### Step 4 — Drop `WD2M` (and the bookkeeping columns YEAR, DOY)

**WHAT:** remove `WD2M` entirely; also remove `YEAR` and `DOY`, which are fully contained in the date index.

**WHY:** WD2M is invalid on every day from 2012-12-31 (Stage 0/1). As approved, we do not re-download wind components, so the
analysis stays reproducible from the single raw file. YEAR/DOY are dropped so that "calendar year" cannot accidentally enter
a model as a feature (it would let a model learn the 2016 regime as a function of time, which does not generalise).

**Alternatives considered:** setting WD2M to NaN after 2012 and imputing — rejected (13 fabricated years, Stage 1 Q3).

**Look for:** 12 columns remain.

In [4]:
before_cols = list(clean.columns)
clean = clean.drop(columns=["WD2M", "YEAR", "DOY"])
print("dropped:", sorted(set(before_cols) - set(clean.columns)))
print("kept   :", list(clean.columns))

dropped: ['DOY', 'WD2M', 'YEAR']
kept   : ['QV2M', 'RH2M', 'IMERG_PRECTOT', 'T2M', 'T2M_MAX', 'T2M_MIN', 'T2MDEW', 'PS', 'WS2M', 'WS10M', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR']


**Interpretation:** WD2M, YEAR and DOY are removed; 12 variables remain: the 10 Set-A/B variables plus
`ALLSKY_SFC_SW_DWN` and `IMERG_PRECTOT` for the Stage-6 ablations.

**Decision:** WD2M is gone from every downstream file.

**Paper note:** *2-m wind direction was excluded from the analysis because its values were physically invalid for the
entire 2013–2025 period.*

### Step 5 — Limited past-only gap filling (max 2 days) and the before/after missing-value table

**WHAT:** forward-fill **interior** gaps of at most 2 consecutive days: `ffill(limit=2, limit_area="inside")`
(verified in the installed pandas 3.0.5). Leading NaN blocks (IMERG before 1998, radiation before 1984) are **not** touched,
because there is no earlier value to carry forward and they are not "rare gaps".

**WHY past-only?** A forward fill uses only the last *earlier* observation, i.e. information that existed on that day.
**Interpolation** (`interpolate()`) or **back-fill** would use a *later* value: a filled day t would contain information from
day t+1 or later — exactly what our target is. If rain on day t+1 were used to fill humidity on day t, the model could
"see the future" through the filled value. The 2-day limit prevents long artificial flat stretches.

**Alternatives considered:** linear interpolation (leaks future), seasonal-climatology fill (needs to be fitted on training
data only — overkill when there are no gaps), model-based imputation inside the Pipeline (later, if ever needed).

**Look for:** how many cells were actually filled — Stage 0 found no interior gaps, so we expect 0.

In [5]:
before = clean.isna().sum()
clean = clean.ffill(limit=2, limit_area="inside")
after = clean.isna().sum()
handling = pd.Series("complete; no action", index=clean.columns)
handling["ALLSKY_SFC_SW_DWN"] = "kept with NaN 1981-1983; not in Set A/B; Stage-6 ablation from 1984"
handling["IMERG_PRECTOT"] = "kept with NaN 1981-1997; never a predictor; secondary target 1998+"
miss_tab = pd.DataFrame({"NaN_before": before, "NaN_after": after, "filled": before - after,
                         "first_valid": [str(clean[c].first_valid_index().date()) for c in clean],
                         "handling": handling})
miss_tab.loc["WD2M (dropped)"] = [0, "-", "-", "invalid from 2012-12-31 (4,749 days)", "dropped (approved plan, item 1)"]
miss_tab.to_csv(TAB / "02_missing_before_after_cleaning.csv")
miss_tab

,NaN_before,NaN_after,filled,first_valid,handling
QV2M,0,0,0,1981-01-01,complete; no action
RH2M,0,0,0,1981-01-01,complete; no action
IMERG_PRECTOT,6209,6209,0,1998-01-01,kept with NaN 1981-1997; never a predictor; se...
T2M,0,0,0,1981-01-01,complete; no action
T2M_MAX,0,0,0,1981-01-01,complete; no action
T2M_MIN,0,0,0,1981-01-01,complete; no action
T2MDEW,0,0,0,1981-01-01,complete; no action
PS,0,0,0,1981-01-01,complete; no action
WS2M,0,0,0,1981-01-01,complete; no action
WS10M,0,0,0,1981-01-01,complete; no action


**Interpretation:** **0 cells were filled** — as Stage 0 showed, there are no interior gaps. The 2-day past-only fill is
therefore a documented safeguard that did nothing on this dataset (good: no data were invented). The only NaN are the two
leading blocks, which are **kept as NaN**:
- `ALLSKY_SFC_SW_DWN` (1,095 days, 1981–1983): excluded from Sets A and B; usable from 1984 in a Stage-6 ablation.
- `IMERG_PRECTOT` (6,209 days, 1981–1997): never a predictor for PRECTOTCORR; secondary target from 1998.

**Decision:** clean data = parsed data minus WD2M/YEAR/DOY; table saved to `reports/tables/02_missing_before_after_cleaning.csv`.

**Paper note:** *No gap filling was required for the variables used as predictors; a past-only forward fill limited to two
days was specified as a safeguard but affected no observations.*

### Step 6 — Save `clean_daily.parquet`

**WHAT:** save the cleaned table and reload it to verify the round-trip.

**WHY:** Stage 3+ notebooks read from this file. Keeping `ALLSKY_SFC_SW_DWN` and `IMERG_PRECTOT` (with their NaN blocks)
preserves them for the Stage-6 ablations without letting them leak into the main feature sets.

**Alternatives considered:** saving only Set-A/B variables — would force a later notebook to go back to the parsed file.

**Look for:** 16,436 × 12, identical after reload.

In [6]:
clean.to_parquet(PROC / "clean_daily.parquet")
pd.testing.assert_frame_equal(pd.read_parquet(PROC / "clean_daily.parquet"), clean, check_freq=False)
print("saved clean_daily.parquet:", clean.shape, clean.index.min().date(), "->", clean.index.max().date())

saved clean_daily.parquet: (16436, 12) 1981-01-01 -> 2025-12-31


**Interpretation:** `clean_daily.parquet` (16,436 × 12) is saved and reloads identically.

**Decision:** it is the single input for feature construction and for the Stage-6 ablations.

**Paper note:** *The cleaned daily dataset contained 16,436 days and twelve variables.*

# Part 2 — Prompt 2.2: targets and features

### Step 7 — Targets for day t+1

**WHAT:** `F.build_targets` creates, for each row t, from `PRECTOTCORR` on day **t+1** (`shift(-1)`):

| target | definition | task |
|---|---|---|
| `y_cls` | 1 if rain(t+1) ≥ 1.0 mm | primary classification |
| `y_cls_01` | 1 if rain(t+1) ≥ 0.1 mm | sensitivity |
| `y_cls_25` | 1 if rain(t+1) ≥ 2.5 mm | sensitivity |
| `y_reg` | rain(t+1) in mm | regression |
| `y_reg_log` | log(1 + rain(t+1)) | regression on a less skewed scale |

**WHY:** each row then holds "everything known at the end of day t" (features) and "what happened next day" (targets) —
a standard supervised layout. `shift(-1)` equals "next calendar day" only because the index is a gap-free daily index
(asserted in Step 3 and tested again in Part 3).

**Alternatives considered:** joining on `date + 1 day` instead of `shift(-1)` — equivalent here; we use it as an
independent check in the target-shift test.

**Look for:** class balance of the three binary targets over the training rows.

In [7]:
Y_all = F.build_targets(clean)
print("last row target (should be NaN, no day 2026-01-01):", Y_all.iloc[-1].to_dict())
Y_all.loc[:TRAIN_LAST_ROW].describe().T[["count", "mean", "std", "min", "50%", "max"]]

last row target (should be NaN, no day 2026-01-01): {'y_reg': nan, 'y_reg_log': nan, 'y_cls': nan, 'y_cls_01': nan, 'y_cls_25': nan}


,count,mean,std,min,50%,max
y_reg,14243.0,7.652,14.852,0.0,0.910,288.100
y_reg_log,14243.0,1.199,1.320,0.0,0.647,5.667
y_cls,14243.0,0.494,0.500,0.0,0.000,1.000
y_cls_01,14243.0,0.594,0.491,0.0,1.000,1.000
y_cls_25,14243.0,0.423,0.494,0.0,0.000,1.000


**Interpretation:** the last day (2025-12-31) has no target, as expected. Over the training rows the targets are:
`y_cls` 49.4 % positive, `y_cls_01` 59.4 %, `y_cls_25` 42.3 %; `y_reg` mean 7.65 mm, median 0.91 mm, max 288 mm;
`y_reg_log` reduces the range from 0–288 to 0–5.7 and the std/mean ratio from ~1.9 to ~1.1.

**Decision:** five targets fixed. Note the regression target is extremely skewed: report MAE and RMSE, and compare models
trained on `y_reg` vs `y_reg_log` (back-transformed with `expm1` for evaluation in mm — this back-transform estimates something
closer to the median, not the mean; we will discuss this in Stage 3).

**Paper note:** *Targets were defined from precipitation on day t + 1: three binary rain-occurrence indicators (≥ 0.1, 1.0
and 2.5 mm) and the precipitation amount on its original and log(1 + x) scales.*

### Step 8 — Build Feature Sets A and B and align their rows

**WHAT:** `F.build_features(clean, "A")` and `(clean, "B")`, then `F.assemble` keeps only rows where every A feature, every
B feature and the target exist. The families (all computed from rows ≤ t):

| family | definition | Set A variables | Set B adds |
|---|---|---|---|
| current | value on day t; `rain_log1p` = log(1+mm); `wet_today` | PRECTOTCORR, T2M, T2M_MAX, T2M_MIN, RH2M, WS2M | PS, QV2M, T2MDEW, WS10M |
| lag | value on day t−k, k ∈ {1,2,3,5,7} (rain as log1p) | all of the above | all of the above |
| rolling mean | mean over the last 3/7/14/30 days incl. t | all except rain | all except rain |
| rolling rain sum | log(1 + rain sum over the last 3/7/14/30 days) | yes | — |
| spell | `wet_run_len`, `days_since_rain` (≥ 1 mm) | yes | — |
| tendency | x(t) − x(t−1) | T2M, RH2M | PS, T2MDEW, QV2M |
| derived | DTR = T2M_MAX − T2M_MIN | yes | DPD = T2M − T2MDEW |
| season | sin/cos of day-of-year (1st and 2nd harmonics), month | yes | — |

`ALLSKY_SFC_SW_DWN` and `IMERG_PRECTOT` are in neither set (approved plan). No calendar year.

**WHY both sets share one index:** rule 7 — every model and both feature sets are compared on exactly the same rows, so any
score difference comes from the features, not from different days. The warm-up (first 29 days: the 30-day windows are not
yet full) and the last day (no t+1) are dropped once, for both.

**Alternatives considered:** `min_periods=1` in rolling windows to keep the first 29 days — the first rows would then be
computed from fewer days than later rows (a different feature definition); losing 29 of 16,436 days is negligible.

**Look for:** A and B with the same number of rows, identical index, no NaN.

In [8]:
XA, XB, Y, dict_a, dict_b = F.assemble(clean)
assert XA.index.equals(XB.index) and XA.index.equals(Y.index)
assert not XA.isna().any().any() and not XB.isna().any().any()
dropped = clean.index.difference(XA.index)
print(f"Set A: {XA.shape} | Set B: {XB.shape} | rows {XA.index.min().date()} -> {XA.index.max().date()}")
print(f"dropped {len(dropped)} rows: {dropped.min().date()}..{dropped[-2].date()} (warm-up) and {dropped[-1].date()} (no t+1)")
w = clean["PRECTOTCORR"] >= F.WET_MM
first_change = w.ne(w.iloc[0]).idxmax()
print("first wet/dry state change:", first_change.date(), "-> spell counters exact after it:", first_change < XA.index.min())

Set A: (16406, 71) | Set B: (16406, 115) | rows 1981-01-30 -> 2025-12-30
dropped 30 rows: 1981-01-01..1981-01-29 (warm-up) and 2025-12-31 (no t+1)
first wet/dry state change: 1981-01-07 -> spell counters exact after it: True


**Interpretation:** Set A has **71** and Set B **115** features on exactly the **same 16,406 rows** (1981-01-30 → 2025-12-30),
no NaN. 30 rows were dropped: the 29 warm-up days of the 30-day windows and the last day. The first wet/dry change happens on
1981-01-07, inside the warm-up, so the spell counters are exact on every kept row (no run is cut off by the start of the data).

**Decision:** both sets use the same rows and folds in every later model (rule 7).

**Paper note:** *Two nested feature sets were constructed (A: 71 features from six core variables; B: 115 features adding
pressure, specific humidity, dew point and 10-m wind), evaluated on an identical set of 16,406 daily samples.*

### Step 9 — Feature dictionary

**WHAT:** one table listing every feature: name, definition, family, and whether it is in Set A and/or B.

**WHY:** a reviewer must be able to reproduce every column from its definition; the table also goes into the paper's
supplementary material.

**Alternatives considered:** documenting features only in code comments — not readable for a reviewer.

**Look for:** feature counts per family and set.

In [9]:
fd = dict_b.copy()
fd["set_A"] = fd["feature"].isin(dict_a["feature"])
fd["set_B"] = True
assert set(dict_a["feature"]) <= set(dict_b["feature"])      # A is a subset of B
fd.to_csv(TAB / "02_feature_dictionary.csv", index=False)
display(fd.groupby("family")[["set_A", "set_B"]].sum().astype(int).assign(
        **{"B only": lambda t: t["set_B"] - t["set_A"]}).sort_values("set_B", ascending=False))
fd.head(12)

,set_A,set_B,B only
family,,,
lag,30,50,20
rolling mean,20,36,16
current,7,11,4
season,5,5,0
tendency,2,5,3
rolling rain sum,4,4,0
derived,1,2,1
spell,2,2,0


,feature,definition,family,set_A,set_B
0,T2M,T2M on day t,current,True,True
1,T2M_MAX,T2M_MAX on day t,current,True,True
2,T2M_MIN,T2M_MIN on day t,current,True,True
3,RH2M,RH2M on day t,current,True,True
4,WS2M,WS2M on day t,current,True,True
5,PS,PS on day t,current,False,True
6,QV2M,QV2M on day t,current,False,True
7,T2MDEW,T2MDEW on day t,current,False,True
8,WS10M,WS10M on day t,current,False,True
9,rain_log1p,rain_log1p on day t = log(1 + PRECTOTCORR mm),current,True,True


**Interpretation:** the dictionary lists all 115 features; Set A ⊂ Set B. Most features are lags (50) and rolling means (36);
44 features are in B only. The dictionary is saved to `reports/tables/02_feature_dictionary.csv`.

**Decision:** the dictionary goes into the supplementary material. With 115 partly collinear features (e.g. QV2M and T2MDEW
carry nearly the same information), regularised linear models and trees are appropriate; plain linear regression coefficients
will not be interpretable individually.

**Paper note:** *A complete feature dictionary with definitions is provided in the supplementary material (Table S1).*

### Step 10 — Why each feature family may help predict next-day rain

- **Current values (day t):** the atmosphere is persistent; today's humidity, temperature and rain describe the air mass
  that will still be over Sylhet tomorrow. Today's rain is the persistence signal (Stage 1: P(wet|wet) = 0.85).
- **Lags (t−1 … t−7):** capture *how* the situation developed. PACF showed direct information at rain lag 1–2 and pressure
  lags 2–3. Using log(1+mm) for rain lags stops a single 150 mm day from dominating linear models.
- **Rolling means (3–30 days):** smooth day-to-day noise and describe the regime — e.g. a 14-day humid spell signals monsoon
  onset better than one humid day. The 30-day window tracks the slow seasonal moisture build-up.
- **Rolling rain sums (3–30 days):** soil/atmosphere "wetness memory" and the active/break phases of the monsoon
  (multi-week wet and dry periods).
- **Spell memory:** the chance that a wet spell continues or a dry spell ends depends on how long it has lasted; trees can
  use these counters directly, and they encode information that 7 lags cannot (spells can last > 100 days).
- **Tendencies (1-day change):** falling pressure, rising humidity or dew point, and cooling are classic precursors of
  rain-bearing systems (nor'westers in the pre-monsoon, monsoon depressions). Stage 1 PACF of PS supported this.
- **Derived — DTR:** a small diurnal temperature range indicates cloudy, moist days (clouds limit daytime heating and
  night-time cooling). **DPD:** a small dew-point depression means air close to saturation, so little lifting is needed to
  produce rain.
- **Season (sin/cos, month):** lets models condition every relationship on the time of year (Stage 1: PS correlates −0.71
  overall but −0.10 within the monsoon). sin/cos make 31 December and 1 January neighbours; the 2nd harmonic allows the
  asymmetric shape of the monsoon (fast onset, slower retreat). `month` is an integer here; linear models will one-hot encode
  it inside their Pipeline (rule 6), trees can use it directly.

### Step 11 — Save `features_A.parquet` and `features_B.parquet`

**WHAT:** save each feature set together with the five target columns (prefixed `y_`), so a later notebook reads one file.

**WHY:** storing targets with features guarantees row alignment. Models must select features with
`[c for c in df if not c.startswith("y_")]`.

**Alternatives considered:** separate `targets.parquet` — another alignment step later, easy to get wrong.

**Look for:** shapes and that both files share the same index.

In [10]:
for name, X in (("A", XA), ("B", XB)):
    out = PROC / f"features_{name}.parquet"
    X.join(Y).to_parquet(out)
    back = pd.read_parquet(out)
    print(f"features_{name}.parquet: {back.shape} ({X.shape[1]} features + {Y.shape[1]} targets)")
print("identical index:", pd.read_parquet(PROC / "features_A.parquet").index.equals(pd.read_parquet(PROC / "features_B.parquet").index))

features_A.parquet: (16406, 76) (71 features + 5 targets)


features_B.parquet: (16406, 120) (115 features + 5 targets)


identical index: True


**Interpretation:** both files are saved with their 5 target columns; they share one index.

**Decision:** downstream rule: features = columns not starting with `y_`.

**Paper note:** *Feature matrices and targets were stored together to guarantee alignment of predictors and outcomes.*

# Part 3 — Prompt 2.3: proving there is no leakage

### Step 12 — (a) Truncation test, with a deliberately leaky feature as negative control

**WHAT:** pick 3 random cut-off dates d (seeded). Rebuild all features from `clean.loc[:d]` — as if the data ended on day
d — and require that, for every date ≤ d, every feature value is **identical** to the full-data version.
Then repeat with a deliberately leaky feature (a *centred* 3-day rolling mean, which uses day t+1) to show the test catches it.

**WHY / what it catches:** if any feature at row t used a row > t (a centred window, a negative shift, a back-fill, a
normalisation over the whole series, a climatology computed from all years), truncating the future would change its value
on or before d. Identical values prove the features are computable at the end of day t.

**Alternatives considered:** only reading the code — reviewers and future-you make mistakes; an executable test re-runs
every time the feature code changes (it is also in `tests/test_leakage.py` for `pytest`).

**Look for:** `PASS` for A and B at all 3 dates, and `FAIL` for the leaky control.

In [11]:
def truncation_test(builder, dates):
    full = builder(clean)
    out = {}
    for d in dates:
        part = builder(clean.loc[:d])
        a, b = full.loc[:d], part.loc[:d]
        same = a.index.equals(b.index) and a.equals(b)
        out[str(d.date())] = "PASS" if same else f"FAIL ({int((a.ne(b) & ~(a.isna() & b.isna())).sum().sum())} cells differ)"
    return out

idx = clean.index[100:-100]
dates = sorted(pd.DatetimeIndex(rng.choice(idx, size=3, replace=False)))
leaky = lambda c: F.build_features(c, "A")[0].assign(
    LEAKY_rain_centred3=np.log1p(c["PRECTOTCORR"]).rolling(3, center=True).mean())
trunc = pd.DataFrame({"Set A": truncation_test(lambda c: F.build_features(c, "A")[0], dates),
                      "Set B": truncation_test(lambda c: F.build_features(c, "B")[0], dates),
                      "Set A + leaky control": truncation_test(leaky, dates)})
trunc.to_csv(TAB / "02_leakage_truncation_test.csv")
trunc

,Set A,Set B,Set A + leaky control
1985-03-29,PASS,PASS,FAIL (1 cells differ)
2010-05-16,PASS,PASS,FAIL (1 cells differ)
2015-09-05,PASS,PASS,FAIL (1 cells differ)


**Interpretation:** at the 3 random cut-off dates (1985-03-29, 2010-05-16, 2015-09-05) every feature of Set A and Set B is
**identical** with and without the future data → **PASS**. The leaky control **FAILS**, and notice *how*: exactly **1 cell**
differs — the centred mean on the cut-off day itself, the only day whose window reaches into the removed future. A leak can be
this small, so the test must compare every value up to *and including* d.

**Decision:** features are computable at the end of day t (rule 5 verified, not just intended).

**Paper note:** *A truncation test confirmed that all features at date t were invariant to the removal of observations after
t, while a deliberately leaky centred-window feature was detected by the same test.*

### Step 13 — (b) Target-shift test and "no feature equals the target"

**WHAT:**
1. For every row t, look up rain on the calendar date t + 1 day **by date** (not by position) and require `y_reg` to equal it;
   check `y_reg_log = log1p(y_reg)` and each `y_cls*` = (y_reg ≥ threshold).
2. For every feature, check that it is not identical (or perfectly rank-correlated) to any target, nor to rain on day t+1
   in any of the forms used by the features (mm, log1p, wet flag).

**WHY / what it catches:** (1) catches a wrong shift direction (`shift(+1)` would predict *yesterday*), off-by-one errors
and hidden gaps in the index. (2) catches a target copied into the feature table by construction (e.g. a feature built
from `shift(-1)` by mistake), which would give near-perfect, meaningless scores (rule 11).

**Alternatives considered:** checking only a few rows by eye — not exhaustive; here every row is checked.

**Look for:** all checks `True`; maximum |Spearman ρ| between any feature and any target-like series well below 1.

In [12]:
rain = clean["PRECTOTCORR"]
next_by_date = rain.reindex(Y.index + pd.Timedelta(days=1)).to_numpy()
checks = {"y_reg == rain(date + 1 day), all rows": np.array_equal(Y["y_reg"].to_numpy(), next_by_date),
          "y_reg_log == log1p(y_reg)": np.allclose(Y["y_reg_log"], np.log1p(Y["y_reg"])),
          **{f"{k} == (y_reg >= {t})": (Y[k] == (Y["y_reg"] >= t).astype(int)).all() for k, t in F.THRESHOLDS.items()}}

targets_like = Y.assign(next_rain_log1p=np.log1p(Y["y_reg"]), next_wet=(Y["y_reg"] >= 1).astype(int))
tl_tr = targets_like.loc[:TRAIN_LAST_ROW]                 # statistics on training rows only (rule 4)
rho = XB.loc[:TRAIN_LAST_ROW].apply(lambda col: tl_tr.corrwith(col, method="spearman").abs().max())
exact = [f for f in XB if any(np.array_equal(XB[f].to_numpy(), targets_like[t].to_numpy()) for t in targets_like)]
checks["no feature identical to a target"] = len(exact) == 0
checks["max |rho| feature vs any target-like < 0.95"] = rho.max() < 0.95
display(pd.Series(checks, name="passed").to_frame())
print("highest |rho| with any target-like series:", rho.idxmax(), round(rho.max(), 3))

,passed
"y_reg == rain(date + 1 day), all rows",True
y_reg_log == log1p(y_reg),True
y_cls == (y_reg >= 1.0),True
y_cls_01 == (y_reg >= 0.1),True
y_cls_25 == (y_reg >= 2.5),True
no feature identical to a target,True
max |rho| feature vs any target-like < 0.95,True


highest |rho| with any target-like series: rain_log1p 0.81


**Interpretation:** every row's `y_reg` equals rain on the **calendar** next day (looked up by date, not position);
`y_reg_log` and the three `y_cls*` targets are consistent; no feature is identical to any target or to next-day rain in mm,
log1p or wet-flag form. The highest |ρ| between any feature and any target-like series is **0.81** (`rain_log1p` — today's
rain), which is the persistence relationship measured in Stage 1, not a copy of the target.

**Decision:** target construction is verified.

**Paper note:** *Targets were verified to correspond exactly to the following calendar day for all samples, and no feature
was identical to, or perfectly rank-correlated with, any target.*

### Step 14 — (c) Correlation scan on training rows

**WHAT:** Spearman ρ of every Set-B feature with `y_reg` and point-biserial-style Spearman with `y_cls`, on **training rows
only** (feature date ≤ 2019-12-30); list the top 15 and flag |ρ| > 0.9 as suspicious.

**WHY / what it catches:** a leak that the first two tests miss — e.g. a variable that is itself a near-copy of tomorrow's
rain (a product with a different day boundary, or a variable computed from next-day data upstream) — shows up as an
implausibly high correlation. Every high value must have a physical explanation before we accept it.

**Alternatives considered:** mutual information — catches non-monotone relations but is harder to interpret; Spearman is
consistent with Stage 1.

**Look for:** the strongest features should be the persistence/moisture features found in Stage 1 (ρ ≈ 0.7–0.8), with
nothing above 0.9.

In [13]:
trX, trY = XB.loc[:TRAIN_LAST_ROW], Y.loc[:TRAIN_LAST_ROW]
scan = pd.DataFrame({"rho_y_reg": trX.corrwith(trY["y_reg"], method="spearman"),
                     "rho_y_cls": trX.corrwith(trY["y_cls"], method="spearman")})
scan["abs_max"] = scan.abs().max(axis=1)
scan = scan.sort_values("abs_max", ascending=False).join(fd.set_index("feature")["family"])
scan["suspicious(>0.9)"] = scan["abs_max"] > 0.9
scan.round(3).to_csv(TAB / "02_leakage_correlation_scan_train.csv")
print("training rows:", len(trX), "| features with |rho| > 0.9:", int(scan["suspicious(>0.9)"].sum()))
scan.head(15).round(3)

training rows: 14214 | features with |rho| > 0.9: 0


,rho_y_reg,rho_y_cls,abs_max,family,suspicious(>0.9)
rain_log1p,0.810,0.738,0.810,current,False
rain_sum3_log1p,0.760,0.694,0.760,rolling rain sum,False
days_since_rain,-0.758,-0.711,0.758,spell,False
wet_today,0.744,0.704,0.744,current,False
QV2M,0.737,0.695,0.737,current,False
T2MDEW,0.736,0.695,0.736,current,False
wet_run_len,0.735,0.693,0.735,spell,False
doy_cos1,-0.724,-0.686,0.724,season,False
rain_sum7_log1p,0.721,0.665,0.721,rolling rain sum,False
QV2M_mean3,0.721,0.680,0.721,rolling mean,False


**Interpretation:** on the 14,214 training rows **no feature exceeds |ρ| = 0.9**. The strongest ones, each with a physical
explanation:
- `rain_log1p` (0.81), `wet_today` (0.74), `rain_sum3_log1p` (0.76), `days_since_rain` (−0.76), `wet_run_len` (0.74):
  **persistence** of rain and spells (Stage 1: P(wet|wet) = 0.85). ρ = 0.81 reproduces the Stage-1 value exactly — a useful
  cross-notebook consistency check.
- `QV2M`, `T2MDEW` and their short means/lags (≈ 0.71–0.74): atmospheric moisture.
- `doy_cos1` (−0.72): **the seasonal cycle alone** — cos is lowest in mid-year, when it rains. This shows how much skill a
  model can get from the calendar alone; it is what the climatology baseline captures.
- `T2M_MIN_mean3/7` (0.72): warm nights = cloudy, humid monsoon air (largely seasonal, Stage 1).

**Decision:** no suspicious feature. Because single features already reach ρ ≈ 0.8, high overall scores in Stage 3 are
*expected*; the real question will be skill **above persistence and climatology** (rule 11 applies to scores far above
those baselines).

**Paper note:** *No feature exhibited a suspiciously high association with next-day rainfall on the training data
(maximum |ρ| = 0.81, for same-day rainfall), and all strong associations were attributable to rainfall persistence,
atmospheric moisture or seasonality.*

### Step 15 — Run the same leakage tests with `pytest`

**WHAT:** run `tests/test_leakage.py`, which repeats the truncation and target-shift tests on the saved files.

**WHY:** tests in a file can be re-run with one command after *any* change to `src/features.py`, without opening this
notebook — a cheap safety net for the rest of the project.

**Alternatives considered:** notebook-only tests — easy to forget to re-run.

**Look for:** all tests passed.

In [14]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(ROOT / "tests/test_leakage.py")],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1500:], r.stderr[-500:])

..........                                                               [100%]
10 passed in 1.69s
 


**Interpretation:** all **10** pytest tests pass: truncation (2 sets × 3 dates), negative control, calendar-next-day target,
and saved files equal to the code output with no target copied into the features.

**Decision:** run `python -m pytest -q tests/` after any change to `src/features.py`.

**Paper note:** *Leakage tests were automated and re-executed whenever the feature code changed.*

## Stage 2 summary

| Item | Result |
|---|---|
| Rain-day threshold | **1.0 mm primary**; 0.1 and 2.5 mm sensitivity |
| Cleaning | `-999`: 0 left · duplicates: 0 · WD2M, YEAR, DOY dropped · past-only fill (≤ 2 days): **0 cells filled** |
| Kept with NaN, not predictors | ALLSKY_SFC_SW_DWN (1981–1983), IMERG_PRECTOT (1981–1997) |
| Targets | `y_cls`, `y_cls_01`, `y_cls_25`, `y_reg`, `y_reg_log` (day t+1) |
| Feature Set A / B | 71 / 115 features; identical 16,406 rows (1981-01-30 → 2025-12-30) |
| Leakage tests | truncation PASS (3 dates × 2 sets); leaky control detected; target = next calendar day; max train |ρ| = 0.81; pytest 10/10 |

**Outputs:** `data/processed/clean_daily.parquet`, `features_A.parquet`, `features_B.parquet`; `src/features.py`;
`tests/test_leakage.py`; `reports/tables/02_*.csv`.

**Reminder for Stage 3:** training rows = feature dates ≤ 2019-12-30 (target in 2019); validation folds inside 2014–2019
reported separately for 2014–2015 and 2016–2019; test 2020–2025 stays locked.